# Figure S1, Cox model
Mortality by calendar period. Time scale: years since stroke. Follow-up split at period boundaries.
Adjusted for stroke type. Age not in data.
Follow-up per period: full, or cut at 1.5 / 1.75 years after period start (or entry).

In [ ]:
import os
from enum import Enum
from itertools import product
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from lifelines import CoxTimeVaryingFitter

In [ ]:
data_path = '../data/excel_4figures.xlsx'
output_dir = '../data/figures'
results_path = '../data/tables/figureS1_period_results.xlsx'

ONE_YEAR = pd.Timedelta(days=365.25)
LANDMARK_DAYS = 90

# same-day deaths
MIN_DURATION_DAYS = 0.1

# no death date: alive at end of follow-up
END_OF_FOLLOW_UP = pd.Timestamp('2025-01-01')

# (start, end, color); colors from original paper
PERIODS = {
    'Pre-pandemic': (pd.Timestamp('2018-01-01'), pd.Timestamp('2020-01-01'), '#5a7a8c'),
    'Pandemic': (pd.Timestamp('2020-01-01'), pd.Timestamp('2022-01-01'), '#b32f1e'),
    'Post-pandemic': (pd.Timestamp('2022-01-01'), END_OF_FOLLOW_UP, '#95BF8F'),
}
REFERENCE_PERIOD = 'Pandemic'
COVARIATES = [period for period in PERIODS if period != REFERENCE_PERIOD] + ['ICH']

# no difference
REFERENCE_HR = 1


class Entry(Enum):
    DAY_90 = '90-day survivors, from day 90'
    ONSET = 'All patients, from onset'


class FollowUp(Enum):
    FULL = 'full'
    HORIZON_1_5Y = 'horizon_1_5y'
    # 2 years - 90 days: end of pre-pandemic from day 90
    HORIZON_1_75Y = 'horizon_1_75y'


# None: until period end
HORIZON_YEARS = {FollowUp.FULL: None, FollowUp.HORIZON_1_5Y: 1.5, FollowUp.HORIZON_1_75Y: 1.75}

In [ ]:
survival_df = pd.read_excel(data_path).dropna(how='all')

onset = pd.to_datetime(survival_df['Onset date'].astype(str), format='%Y%m%d')
death = pd.to_datetime(survival_df['5Y Death date'], format='%d/%m/%Y')

patients_df = pd.DataFrame({
    'onset': onset,
    'death': death,
    'ich': survival_df['Type of event'] == 'Intracerebral hemorrhage',
})
len(patients_df)

In [ ]:
def select_entry(entry):
    """Entry date per patient; 90-day survivors only when entering at day 90."""
    if entry == Entry.ONSET:
        return patients_df.assign(entry=patients_df['onset'])

    entry_date = patients_df['onset'] + pd.Timedelta(days=LANDMARK_DAYS)
    survivors = ~(patients_df['death'] <= entry_date)
    return patients_df[survivors].assign(entry=entry_date[survivors])


def split_by_period(entry, follow_up):
    """One row per patient and period, times in years since stroke.

       onset   entry      2020-01-01          2022-01-01       death
         |-------|---pre-----|------pandemic------|----post----X
    """
    patients = select_entry(entry)

    died = patients['death'] < END_OF_FOLLOW_UP
    exit_date = patients['death'].where(died, END_OF_FOLLOW_UP)
    exit_date = exit_date.where(exit_date > patients['entry'], patients['entry'] + pd.Timedelta(days=MIN_DURATION_DAYS))

    intervals = []
    for period, (start, end, _) in PERIODS.items():
        interval_start = patients['entry'].where(patients['entry'] > start, start)
        interval_stop = exit_date.where(exit_date < end, end)

        horizon_years = HORIZON_YEARS[follow_up]
        if horizon_years is not None:
            horizon_stop = interval_start + horizon_years * ONE_YEAR
            interval_stop = interval_stop.where(interval_stop < horizon_stop, horizon_stop)

        in_period = interval_start < interval_stop

        intervals.append(pd.DataFrame({
            'id': patients.index,
            'start': (interval_start - patients['onset']) / ONE_YEAR,
            'stop': (interval_stop - patients['onset']) / ONE_YEAR,
            'death': died & (exit_date <= interval_stop),
            'period': period,
            'ICH': patients['ich'].astype(int),
        })[in_period])

    long_df = pd.concat(intervals, ignore_index=True)
    for period in PERIODS:
        if period == REFERENCE_PERIOD:
            continue
        long_df[period] = (long_df['period'] == period).astype(int)
    return long_df

In [ ]:
def person_years(long_df):
    """Deaths, person-years and crude rate per 100 person-years."""
    table = long_df.assign(years=long_df['stop'] - long_df['start']).groupby('period', sort=False).agg(
        deaths=('death', 'sum'), person_years=('years', 'sum')).reset_index()
    table['rate_per_100'] = 100 * table['deaths'] / table['person_years']
    return table.round(1)


def hazard_ratios(model):
    summary = model.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']]
    summary.columns = ['HR', 'HR lower 95%', 'HR upper 95%', 'p']
    return summary.reset_index().round(3)

def save_sheet(table, sheet_name):
    """Write one sheet, keeping the other notebook's sheets."""
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    if not os.path.exists(results_path):
        table.to_excel(results_path, sheet_name=sheet_name, index=False)
        return

    with pd.ExcelWriter(results_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        table.to_excel(writer, sheet_name=sheet_name, index=False)


def fit_cox(long_df):
    model = CoxTimeVaryingFitter()
    model.fit(long_df[['id', 'start', 'stop', 'death'] + COVARIATES],
              id_col='id', event_col='death', start_col='start', stop_col='stop')
    return model


models = {}
cox_tables = []
person_years_tables = []
for entry, follow_up in product(Entry, FollowUp):
    long_df = split_by_period(entry, follow_up)
    models[(entry, follow_up)] = fit_cox(long_df)

    labels = {'entry': entry.value, 'follow_up': follow_up.value}
    cox_tables.append(hazard_ratios(models[(entry, follow_up)]).assign(
        patients=long_df['id'].nunique(), deaths=int(long_df['death'].sum()), **labels))
    person_years_tables.append(person_years(long_df).assign(**labels))

cox_df = pd.concat(cox_tables, ignore_index=True)
cox_df = cox_df[['entry', 'follow_up'] + [column for column in cox_df if column not in ('entry', 'follow_up')]]
save_sheet(cox_df, 'Cox (adjusted)')

person_years_df = pd.concat(person_years_tables, ignore_index=True)
person_years_df = person_years_df[['entry', 'follow_up'] + [column for column in person_years_df if column not in ('entry', 'follow_up')]]
save_sheet(person_years_df, 'Person-years')

cox_df

In [ ]:
# marker per entry: filled for day 90, open for onset
ENTRY_MARKERS = {Entry.DAY_90: 'full', Entry.ONSET: 'none'}
ENTRY_OFFSET = 0.12

# forest plot: full follow-up
full_models = {entry: models[(entry, FollowUp.FULL)] for entry in Entry}

fig, ax = plt.subplots(1, 1, figsize=(7, 3))

compared_periods = [period for period in PERIODS if period != REFERENCE_PERIOD]
for row, period in enumerate(compared_periods):
    _, _, color = PERIODS[period]

    for offset_sign, (entry, model) in zip([1, -1], full_models.items()):
        hr, lower, upper = model.summary.loc[period, ['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%']]
        ax.errorbar(hr, row + offset_sign * ENTRY_OFFSET, xerr=[[hr - lower], [upper - hr]], fmt='o',
                    color=color, fillstyle=ENTRY_MARKERS[entry], markersize=7, capsize=3)

ax.axvline(REFERENCE_HR, color='black', linestyle=':', linewidth=1)
ax.set_xscale('log')
ax.set_xticks([0.25, 0.5, 1, 2, 4])
ax.set_xticklabels(['0.25', '0.5', '1', '2', '4'])
ax.minorticks_off()

ax.set_yticks(range(len(compared_periods)))
ax.set_yticklabels([f'{period} vs {REFERENCE_PERIOD.lower()}' for period in compared_periods])
ax.invert_yaxis()

# remove upper and right spine
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')

ax.set_xlabel('Hazard ratio (95% CI)')
# legend encodes entry only, so neutral color
legend_handles = [Line2D([], [], marker='o', linestyle='', color='black', fillstyle=ENTRY_MARKERS[entry], label=entry.value)
                  for entry in full_models]
ax.legend(handles=legend_handles, loc='lower right', frameon=False, bbox_to_anchor=(1, 1))
ax.grid(False)

plt.tight_layout()

In [ ]:
os.makedirs(output_dir, exist_ok=True)
fig.savefig(os.path.join(output_dir, 'figureS1_period_cox.pdf'), dpi=600, bbox_inches='tight')
fig.savefig(os.path.join(output_dir, 'figureS1_period_cox.png'), dpi=300, bbox_inches='tight')